# Laboratorio ETL: CSV → Python → SQL Server
**Curso:** Base de Datos Avanzadas y Big Data · **Unidad II:** Business Intelligence · **Semana 7**  
**Saber esencial:** Diseño de procesos ETL · **Duración sugerida:** 2 horas (con BD preparada)  
**Logro de la unidad:** diseñar soluciones de inteligencia de negocios que respondan a necesidades identificadas, utilizando metodologías y herramientas adecuadas.

**Escenario:** Una empresa comercial dispone de `ventas.csv` (500 líneas sintéticas). La gerencia necesita analizar ventas por producto, cliente/ciudad y tiempo. Debe cargarse un Data Mart en SQL Server con `DimCliente`, `DimProducto`, `DimFecha` y `FactVentas`. La ciudad es atributo del cliente.

**Granularidad:** una fila válida de la tabla de hechos equivale a una línea de venta identificada por `IdVenta`.

**Datos del equipo:** Integrantes: _________________________ · Sección: __________ · Fecha: __________

**Material previo:** `ventas.csv`, `Metadata_Ventas.docx`, `Modelo_Dimensional_Ventas_Actualizado.docx` y `DataMartVentas.sql`. Las cuatro tablas deben crearse con el script SQL antes de la carga; no precargue registros.

**Versión:** 1.1 · **Entorno recomendado:** Anaconda `etl_bi`, kernel `Python (ETL BI)`.

## Propósito y desafío
**Reto:** construir un ETL reproducible y justificable que extraiga datos del CSV, detecte anomalías, prepare tres dimensiones y una tabla de hechos, cargue solo filas válidas y concilie resultados.

**Entregables:** notebook ejecutado, decisiones de limpieza, Data Mart poblado, tabla de cuarentena o reporte de rechazos, consultas de verificación y respuestas reflexivas.

**Regla:** no invente precios, clientes, productos o códigos que no se puedan respaldar. Si un identificador de venta aparece en líneas contradictorias, adopte una política explícita y argumentada. **No asuma que se cargarán las 500 líneas.**

## Antes del laboratorio: guía de configuración (obligatoria)

**Documento independiente:** `Guia_Entorno_ETL_Anaconda.docx`. Siga esa guía **en Anaconda Prompt, antes de abrir este cuaderno**, para crear y activar `etl_bi`, instalar dependencias, registrar el kernel y comprobar el controlador ODBC. **No ejecute comandos `conda create` dentro de las celdas del notebook.**

1. En Anaconda Prompt active `conda activate etl_bi` e inicie `jupyter lab` (o Jupyter Notebook, si lo prefiere).
2. En Jupyter seleccione el kernel **Python (ETL BI)**.
3. Compruebe que `ventas.csv` existe en `CSV_PATH` y que `DataMartVentas.sql` se ejecutó previamente en SSMS.
4. Ajuste `SERVIDOR` y `CONTROLADOR` a su máquina. El ejemplo utiliza autenticación integrada de Windows. No incluya contraseñas en el notebook.
5. Ejecute las verificaciones siguientes antes de pasar a los desafíos.

**Nota de seguridad:** `TrustServerCertificate=yes` solo se usa en este ejemplo de práctica local. En producción, configure certificados confiables y la verificación TLS correspondiente.

In [ ]:
# Diagnóstico del entorno: no modifica SQL Server.
import sys
from pathlib import Path
import pandas as pd
import sqlalchemy
import pyodbc

print("Python:", sys.version.split()[0])
print("Ejecutable de Python:", sys.executable)
print("pandas:", pd.__version__)
print("SQLAlchemy:", sqlalchemy.__version__)
print("pyodbc:", pyodbc.version)
print("Controladores ODBC:", pyodbc.drivers())
print("Kernel esperado: Python (ETL BI); confirme además su selección en Jupyter.")


### Comprobación de archivos y preparación de la conexión

Ubique `ventas.csv` en la carpeta de trabajo o actualice `CSV_PATH`. El código siguiente **configura** el motor SQLAlchemy, pero **no realiza todavía una conexión a SQL Server**. Confirme que dispone del controlador ODBC seleccionado.

In [ ]:
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

CSV_PATH = Path("ventas.csv")  # Actualizar según la carpeta
SERVIDOR = r"localhost\SQLEXPRESS"  # Ajustar a su instancia
BASE_DATOS = "DataMartVentas"
CONTROLADOR = "ODBC Driver 18 for SQL Server"  # Revisar con pyodbc.drivers()
# Con Driver 18, TrustServerCertificate=yes facilita prácticas con certificados locales.
# Solo para desarrollo local; en producción configure certificados TLS válidos.
CADENA_ODBC = (
    f"DRIVER={{{CONTROLADOR}}};SERVER={SERVIDOR};DATABASE={BASE_DATOS};"
    "Trusted_Connection=yes;Encrypt=yes;TrustServerCertificate=yes;"
)
engine = create_engine(URL.create("mssql+pyodbc", query={"odbc_connect": CADENA_ODBC}), future=True)
print("Ruta CSV:", CSV_PATH.resolve())
print("Drivers ODBC disponibles:", pyodbc.drivers())
if not CSV_PATH.is_file():
    raise FileNotFoundError(f"No se encontró el archivo CSV: {CSV_PATH.resolve()}")
if CONTROLADOR not in pyodbc.drivers():
    raise RuntimeError(f"No está disponible {CONTROLADOR}. Controladores: {pyodbc.drivers()}")
print("Archivo CSV encontrado y controlador ODBC disponible.")

### Prueba de conectividad y estructura de destino

Esta celda efectúa una conexión de **solo lectura**. La base de datos y sus cuatro tablas se deben haber creado con el script `DataMartVentas.sql` en SSMS. Si falla, revise el nombre de la instancia, autenticación, servicio SQL Server y controlador ODBC antes de continuar.

In [ ]:
tablas_requeridas = {"DimCliente", "DimProducto", "DimFecha", "FactVentas"}
with engine.connect() as conexion:
    nombre_bd = conexion.execute(text("SELECT DB_NAME()")).scalar_one()
    tablas_encontradas = set(pd.read_sql_query(
        text("SELECT TABLE_NAME FROM INFORMATION_SCHEMA.TABLES WHERE TABLE_SCHEMA = 'dbo' AND TABLE_TYPE = 'BASE TABLE'"),
        conexion
    )["TABLE_NAME"])
print("Base de datos:", nombre_bd)
print("Tablas esperadas:", sorted(tablas_requeridas))
print("Tablas encontradas:", sorted(tablas_encontradas & tablas_requeridas))
assert nombre_bd == BASE_DATOS, "La conexión apunta a otra base de datos"
assert tablas_requeridas <= tablas_encontradas, "Faltan tablas: " + str(tablas_requeridas - tablas_encontradas)
print("Conexión y estructura verificadas (solo lectura).")

## Desafío 1. Explorar y diagnosticar (15 min)
Observe tipos de datos, número de filas, valores ausentes, duplicados de `IdVenta`, fechas no estándar, cantidades/precios inválidos y diferencias de escritura. Genere un breve inventario de hallazgos, sin eliminar registros todavía.

**Preguntas:** ¿por qué leer inicialmente como texto? ¿Qué riesgos existen si pandas infiere valores numéricos o fechas prematuramente?

In [ ]:
# TODO: leer el CSV respetando UTF-8 BOM, conservar inicialmente textos y mostrar muestra y forma.
# TODO: realizar el perfilado y registrar tipos, nulos, duplicados y anomalías.

## Desafío 2. Diseñar reglas de calidad y transformación (20 min)
Construya una matriz: **problema → impacto → regla de tratamiento → evidencia**. Establezca reglas para:
- Espacios en blanco, nombres y categorías con variantes de escritura.
- Fechas ISO y fechas en otros formatos identificados; evite conversiones ambiguas.
- `Cantidad > 0` entero, `PrecioUnitario > 0` numérico, `IdVenta` único.
- Códigos de negocio de cliente y producto y sus atributos descriptivos.
- Valores faltantes que se puedan recuperar mediante evidencia de otros registros de la misma clave; en caso contrario, cuarentena.

**Punto de decisión:** ¿qué hará ante un código de producto desconocido cuya descripción puede corresponder a más de un producto? Justifique.

In [ ]:
# TODO: normalizar campos y crear una estructura con datos aptos para ETL.
# TODO: separar registros aptos y registros en cuarentena con motivo de rechazo.

## Desafío 3. Preparar el modelo dimensional (20 min)
A partir de las filas aceptadas construya:
- `DimCliente` — una fila por `IdCliente`, con nombre y ciudad normalizados.
- `DimProducto` — una fila por `IdProducto`, con producto y categoría normalizados.
- `DimFecha` — una fila por fecha, con clave `AAAAMMDD` y atributos calendario.

Explique cómo resolverá las claves sustitutas `ClienteKey` y `ProductoKey` al poblar `FactVentas`.

In [ ]:
# TODO: construir DataFrames DimCliente, DimProducto, DimFecha y datos de hechos.
# TODO: comprobar la unicidad de cada dimensión y la granularidad de hechos.

## Desafío 4. Cargar SQL Server de forma segura (25 min)
Ejecute `DataMartVentas.sql` antes de esta fase. Cargue en orden dimensiones → hechos. Utilice transacciones de base de datos y evite una segunda inserción del mismo `IdVentaOrigen`.

**Advertencia:** `FactVentas` usa claves sustitutas generadas por SQL Server; no asuma que el orden o valor de los `IDENTITY` coincide con el orden de DataFrames. Recupere el mapeo real de claves.

In [ ]:
# TODO: probar conexión y consultar si el destino está preparado.
# TODO: insertar dimensiones y resolver claves sustitutas desde SQL Server.
# TODO: insertar hechos de forma controlada; evitar duplicados en reejecuciones.

## Desafío 5. Validar y explicar resultados (15 min)
Verifique: filas fuente = filas aceptadas + filas en cuarentena; ausencia de claves foráneas huérfanas; unicidad de `IdVentaOrigen`; coincidencia de cantidades e importes; estabilidad de una segunda ejecución.

Realice **dos consultas**: importe por categoría y ventas por ciudad del cliente. Explique por qué esta última no equivale necesariamente a ciudad de venta.

In [ ]:
# TODO: ejecutar verificaciones SQL con pandas.read_sql_query(..., engine)
# TODO: comparar los resultados con el reporte de ETL.

## Cierre: preguntas crítico-reflexivas
1. ¿Qué anomalía podría provocar el mayor error en un dashboard y por qué?
2. ¿Qué consecuencias tendría reemplazar automáticamente precios no válidos por cero?
3. ¿Por qué `Ciudad` permanece en `DimCliente` en este primer laboratorio? ¿Cuándo sería mejor `DimCiudad` independiente?
4. ¿Cómo demostraría que repetir la carga no altera las ventas?
5. ¿Qué evidencia mostraría a un auditor para justificar registros rechazados?

**Reto opcional:** documente cómo implementar una carga incremental conservando registros históricos y proponiendo una estrategia para cambios de ciudad de clientes.

**Evidencias para entrega:** notebook ejecutado, resumen de reglas ETL, tabla/reporte de cuarentena, consultas SQL y respuestas. Declare el uso de IA, si lo hubo, y explique qué validó personalmente.